# 739. 每日温度

给定一个整数数组 temperatures ，表示每天的温度，返回一个数组 answer ，其中 answer[i] 是指对于第 i 天，下一个更高温度出现在几天后。如果气温在这之后都不会升高，请在该位置用 0 来代替。

 
示例 1:
```text
输入: temperatures = [73,74,75,71,69,72,76,73]
输出: [1,1,4,2,1,1,0,0]
```
示例 2:
```text
输入: temperatures = [30,40,50,60]
输出: [1,1,1,0]
```
示例 3:
```text
输入: temperatures = [30,60,90]
输出: [1,1,0]
```

## 核心心法：谁在候车室里？

在《每日温度》这道题里，我们要找的是“下一个更高的温度”。
当你遍历数组时，如果今天很冷，你不知道未来哪天会变热。所以，今天只能作为一个“悬而未决的任务”，进入候车室等待。

让我们推演一下候车室里的生态：

1. 第 2 天温度 75，它没找到比自己高的，进入候车室。
2. 第 3 天温度 71，它也没找到比自己高的（它比 75 还要冷），进入候车室。
3. 第 4 天温度 69，依然没找到，进入候车室。

此时，看看你的候车室里坐着谁：`[75, 71, 69]`。
发现了吗？**候车室里的温度，天然形成了一个严格的“单调递减”序列！**

为什么绝对不可能出现递增（比如 `[75, 69, 71]`）？
因为如果第 4 天是 71，那第 3 天的 69 **早就等到了比它高的温度，直接结算走人了**，根本没资格继续留在候车室里！

---

## 破局三步曲：面对新题如何套用？

以后遇到任何题目，只要是在一个数组里“寻找下一个更大/更小的元素”，直接祭出以下三步：

**第一步：确定“结算条件”（你要找什么？）**

* 题目要求找“下一个**更大**的元素”，那么新来的元素必须**大于**栈顶元素，才能触发结算。
* 题目要求找“下一个**更小**的元素”，那么新来的元素必须**小于**栈顶元素，才能触发结算。

**第二步：反推“栈内单调性”（候车室里是什么样？）**

* 记住一个铁律：**你要找什么，栈内的单调性就是相反的。**
* 找更大 $\rightarrow$ 还没找到的元素只能越来越小 $\rightarrow$ **单调递减栈**。
* 找更小 $\rightarrow$ 还没找到的元素只能越来越大 $\rightarrow$ **单调递增栈**。

**第三步：确定栈里存什么数据？**

* 如果题目要求算“距离/天数”，栈里**必须存数组的索引（Index）**，因为有了索引，既可以通过 `nums[index]` 拿到值，又能通过索引相减算距离。
* 如果题目只要那个具体的数值，存数值或索引都可以。

In [ ]:
from typing import List

class Solution:
    def dailyTemperatures(self, temperatures: List[int]) -> List[int]:
        n = len(temperatures)
        ans = [0] * n  # 初始化答案数组，默认全为0（如果等不到高温，正好是0）
        
        stack = []     # 单调栈（候车室），专门用来存放“还没有找到更高温度的日子”的【索引】

        # 遍历每一天
        for i, t in enumerate(temperatures):
            
            # 【结算阶段】
            # 如果候车室里有人（栈不为空）
            # 并且今天的温度 t，比候车室里最后进去的那个人（栈顶索引对应的温度）还要高！
            while stack and t > temperatures[stack[-1]]:
                # 栈顶的这个人终于等到了高温，让他出列（弹出）
                prev_index = stack.pop()
                # 计算他等了多少天，并记录到答案数组中
                ans[prev_index] = i - prev_index
            
            # 【等待阶段】
            # 不管刚才有没有结算别人，今天自己也成了一个悬而未决的任务。
            # 乖乖带着自己的索引进入候车室，等待未来某一天的高温来解救自己。
            stack.append(i)
            
        return ans

In [ ]:
# 原始代码 方便理解

class Solution:
    def dailyTemperatures(self, temperatures: List[int]) -> List[int]:
        from collections import deque
        ans = [0 for _ in temperatures]
        # 单调递减栈
        q = deque()
        for index, tem in enumerate(temperatures):
            if not q or tem <= temperatures[q[-1]]:
                q.append(index)
            else:
                while q and tem > temperatures[q[-1]]:
                    day = q.pop()
                    ans[day] = index - day
                q.append(index)
        return ans